# Week 11 — LLMs: Fine-tuning & Prompting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aofphy/SCI193611_ARTIFICIAL_INTELLIGENCE/blob/main/labs/w11_llm_finetune_prompt.ipynb)

**Objective:** ใช้ LLM ผ่าน Hugging Face — prompting และ fine-tuning แบบ LoRA.

> ⚠️ โน้ตบุ๊กนี้ต้องดาวน์โหลดโมเดลและควรใช้ GPU — รันบน **Google Colab (GPU runtime)**. โค้ดเป็น reference ที่ยังไม่ได้รันในรีโพนี้.


## 1) Setup


In [ ]:
# Requires internet + (ideally) GPU. Run on Google Colab with a GPU runtime.
# !pip install -q transformers datasets accelerate peft
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

## 2) Few-shot prompting


In [ ]:
model_name = "distilgpt2"   # small enough to run on CPU/Colab
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def complete(prompt, max_new_tokens=40):
    ids = tok(prompt, return_tensors="pt").input_ids
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

# Few-shot prompting
prompt = ("Translate English to French:\n"
          "sea otter => loutre de mer\n"
          "cheese =>")
print(complete(prompt))

## 3) LoRA fine-tuning (sketch)


In [ ]:
# Parameter-efficient fine-tuning (LoRA) sketch
from peft import LoraConfig, get_peft_model

lora = LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"], lora_dropout=0.05)
peft_model = get_peft_model(model, lora)
peft_model.print_trainable_parameters()
# -> then train with transformers.Trainer on your dataset (see datasets library).

## 4) TODO
- เตรียม dataset ด้วย `datasets` แล้วเทรนด้วย `Trainer`
- เทียบผลก่อน/หลัง fine-tune
- ทดลอง prompting หลายแบบ (zero-shot / few-shot / chain-of-thought)
